#### Jointure spikes/LFP

LFP_spikes/
├── P119_FM71_stim4/
│   ├── P119_FM71_stim4_common_trials.csv
│   └── P119_FM71_stim4_common_metadata.json
├── ...
├── run_all_common_preprocess_sessions.csv
└── run_all_common_preprocess_summary.json

##### une session

In [ ]:
from utils_spike_lfp.spike_lfp_common_preprocess_session import (
    CommonPreprocessConfig,
    prepare_common_session,
    save_common_session_bundle,
    load_common_session_bundle,
    validate_common_trials,
)

micro_root="/media/aube/Aube/Spike-sorting"  # contient Data_folders/...
macro_root="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP" # contient .TRC + events macro
hilbert_root="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP/theta_gamma_cog/results_hilbert" # optionnel
common_root="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP_spikes" 

cfg = CommonPreprocessConfig(
    micro_root=micro_root,
    macro_root=macro_root,
    hilbert_root=hilbert_root,
    pre_length=3.0,
    post_length=3.0,
    epsilon=0.1,
    event_match_strategy="auto",
    compute_micro_macro_offset_qc=True,
)

bundle = prepare_common_session(
    patient="P119_FM71",
    session=4,
    cfg=cfg,
    hilbert_bands=("theta", "alpha", "beta", "low_gamma", "high_gamma"),
)

qc = validate_common_trials(bundle.trials)
print(qc)

session_common_dir = save_common_session_bundle(bundle, common_root)

bundle2 = load_common_session_bundle(
    session_common_dir,
    load_hilbert=True,
    hilbert_bands=("theta", "alpha", "beta")
)



=== Base commune spike-LFP | P119_FM71_stim4 ===
[INFO] offset micro↔macro: offset=-41.471000s, MAD résidus=0s, max|résidu|=1.13687e-13s, n_used=31/31
[INFO] Hilbert exports chargés: ['theta', 'alpha', 'beta', 'low_gamma', 'high_gamma']
{'n_trials': 31, 'missing_required_columns': [], 'nan_counts': {'t_start_micro': 0, 't_end_micro': 0, 't_start_macro': 0, 't_end_macro': 0}, 'n_label_order_mismatch': 0, 'group_counts': {'unknown': 25, 'cog+': 3, 'negatif': 2, 'controle': 1}, 'n_unparsed_stim_labels': 0}


##### toutes les sessions

In [4]:
from utils_spike_lfp.spike_lfp_common_preprocess_batch import (
    CommonBatchPreprocessConfig,
    run_all_common_preprocess,
)
micro_root="/media/aube/Aube/Spike-sorting"  # contient Data_folders/...
macro_root="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP" # contient .TRC + events macro
hilbert_root="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP/theta_gamma_cog/results_hilbert" # optionnel
common_root="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP_spikes" 

cfg_common_batch = CommonBatchPreprocessConfig(
    micro_root=micro_root,
    macro_root=macro_root,
    hilbert_root=hilbert_root,
    common_root=common_root,

    session_source="hilbert",

    hilbert_bands=("theta", "alpha", "beta", "low_gamma", "high_gamma"),
    require_hilbert_exports=True,

    require_existing_nwb=True, # pour ne préparer que les sessions analysables en spikes

    pre_length=3.0,
    post_length=3.0,
    epsilon=0.1,

    event_match_strategy="auto",
    compute_micro_macro_offset_qc=True,

    skip_existing=True,
    verbose=True,
)

out_common = run_all_common_preprocess(cfg_common_batch)
out_common["sessions"].head()

,session,patient,session_num,status,out_dir
0,P119_FM71_stim4,P119_FM71,4,skipped_existing,/home/aube/Documents/article_neuronal_stimic/e...


#### Spike / power

UNE SESSION : 

results_fr_power_corr/
    └── P119_FM71_stim4/                                         {créé}
        ├── P119_FM71_stim4_fr_power_corr_config.json
        ├── P119_FM71_stim4_fr_bins.csv
        ├── P119_FM71_stim4_fr_power_trial_bins_theta.csv
        ├── P119_FM71_stim4_fr_power_trial_bins_alpha.csv
        ├── P119_FM71_stim4_fr_power_trial_bins_beta.csv
        ├── P119_FM71_stim4_fr_power_trial_bins_low_gamma.csv
        ├── P119_FM71_stim4_fr_power_trial_bins_high_gamma.csv
        ├── P119_FM71_stim4_fr_power_correlations.csv
        ├── P119_FM71_stim4_fr_power_corr_run_summary.json
        └── P119_FM71_stim4_fr_power_correlations.csv            {table principale}

MULTI-SESSIONS : 
results_fr_power_corr_all/
├── per_session/
│   ├── P119_FM71_stim4/
│   │   ├── P119_FM71_stim4_fr_power_correlations.csv
│   │   ├── P119_FM71_stim4_fr_power_correlation_signif.csv
│   │   └── figures_significant_histograms/
│   └── ...
├── pooled_across_sessions/
│   ├── ALL_SESSIONS_fr_power_correlations_pooled.csv
│   ├── ALL_SESSIONS_fr_power_correlations_pooled_SIGNIF.csv
│   ├── ALL_SESSIONS_fr_power_pool_summary.json
│   └── figures_significant_histograms/
├── run_all_fr_power_correlations_summary.json
└── run_all_fr_power_correlations_sessions.csv

In [ ]:
#### old

from utils_spike_lfp.spike_lfp_common_preprocess_session import (
    CommonPreprocessConfig,
    prepare_common_session,
    save_common_session_bundle,
    load_common_session_bundle,
    validate_common_trials,
)

from utils_spike_lfp.spike_lfp_common_preprocess_session import load_common_session_bundle, get_nwb
from utils_spike_lfp.spike_lfp_power_correlations import (
    FRPowerCorrelationConfig,
    make_units_dict_from_spikes_object,
    run_session_fr_power_correlations,
)
root_micro = '/media/aube/Aube/'
patient = 'P119_FM71'
session = '4'

# on recharge base commune avec les Hilbert : 
session_common_dir = f"/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP_spikes/{patient}_stim{session}"
bundle = load_common_session_bundle(
    session_common_dir,
    load_hilbert=True,
    hilbert_bands=("theta", "alpha", "beta", "low_gamma", "high_gamma"),
)

# on prépare les spikes depuis le NWB :
spikes = get_nwb(patient, session, root_micro)
units = make_units_dict_from_spikes_object(spikes)

# paramétrage : 
cfg_corr = FRPowerCorrelationConfig(
    bin_width_ms_options=(100.0, 500.0),
    windows_to_correlate=("post",),

    # FR normalisé par baseline pré-stim trial-wise
    fr_norm_method="logratio", # ou alors "zscore"
    epsilon_fr_hz=0.1,

    # LFP = enveloppe/power Hilbert déjà exportée, moyennée dans le bin
    lfp_value_col="lfp_power_bin_mean",

    bands_to_test=("theta", "alpha", "beta", "low_gamma", "high_gamma"),

    localities_to_include=("local", "distant"),

    correlation_methods=("spearman", "pearson"),
    min_trials_for_corr=6,
    min_finite_pairs_for_corr=6,

    correlation_groupings=(
        "all",
        "group_label",
        "locality",
        "group_label_x_locality",
    ),

    save_trial_bin_tables=True,
    save_fr_table=True,
    save_lfp_table=False,
    compression=None,
    verbose=True,
)

out = run_session_fr_power_correlations(
    bundle=bundle,
    units=units,
    out_dir="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP_spikes/results_fr_power_corr",
    cfg=cfg_corr,
)

corr = out["correlations"]
fr_table = out["fr_table"]

corr.head()

/media/aube/Aube/Spike-sorting/Data_folders/P119_FM71/P119_FM71_stim4/ .nwb existe deja ? True
[INFO] P119_FM71_stim4: calcul FR biné pour 5 unités
[INFO] P119_FM71_stim4: binning LFP band=theta
[INFO] P119_FM71_stim4: fusion FR×LFP band=theta
[INFO] P119_FM71_stim4: corrélations band=theta
[INFO] P119_FM71_stim4: binning LFP band=alpha
[INFO] P119_FM71_stim4: fusion FR×LFP band=alpha
[INFO] P119_FM71_stim4: corrélations band=alpha
[INFO] P119_FM71_stim4: binning LFP band=beta
[INFO] P119_FM71_stim4: fusion FR×LFP band=beta
[INFO] P119_FM71_stim4: corrélations band=beta
[INFO] P119_FM71_stim4: binning LFP band=low_gamma
[INFO] P119_FM71_stim4: fusion FR×LFP band=low_gamma
[INFO] P119_FM71_stim4: corrélations band=low_gamma
[INFO] P119_FM71_stim4: binning LFP band=high_gamma
[INFO] P119_FM71_stim4: fusion FR×LFP band=high_gamma
[INFO] P119_FM71_stim4: corrélations band=high_gamma
[OK] P119_FM71_stim4: corrélations sauvegardées -> /home/aube/Documents/article_neuronal_stimic/effets_cog/L

,session,unit_id,band,channel_idx,channel_name,channel_shaft,bin_width_ms,window,time_bin_id,bin_index_in_window,...,lfp_value_col,n_trials_total,n_finite_pairs,rho_or_r,p_value,fr_mean,fr_sd,lfp_mean,lfp_sd,q_value_fdr_bh
0,P119_FM71_stim4,0,theta,0,TPp1-TPp2,TPp,100.0,post,30,0,...,lfp_power_bin_mean,31,31,0.082779,0.657972,-0.974195,1.151679,95.871616,65.259855,0.948923
1,P119_FM71_stim4,0,theta,0,TPp1-TPp2,TPp,100.0,post,30,0,...,lfp_power_bin_mean,31,31,0.180809,0.330363,-0.974195,1.151679,95.871616,65.259855,0.999998
2,P119_FM71_stim4,0,theta,1,TPp2-TPp3,TPp,100.0,post,30,0,...,lfp_power_bin_mean,31,31,0.143614,0.440851,-0.974195,1.151679,116.033260,97.528538,0.903478
3,P119_FM71_stim4,0,theta,1,TPp2-TPp3,TPp,100.0,post,30,0,...,lfp_power_bin_mean,31,31,0.267119,0.146311,-0.974195,1.151679,116.033260,97.528538,0.999998
4,P119_FM71_stim4,0,theta,2,TPp5-TPp6,TPp,100.0,post,30,0,...,lfp_power_bin_mean,31,31,0.426496,0.016730,-0.974195,1.151679,92.834835,56.162616,0.784012


In [ ]:
# pour une session : 

from utils_spike_lfp.spike_lfp_common_preprocess_session import load_common_session_bundle

from utils_spike_lfp.spike_lfp_power_correlations import (
    FRPowerCorrelationConfig,
    make_units_dict_from_spikes_object,
    run_session_fr_power_correlations,
)

root_micro = '/media/aube/Aube/'
patient = 'P119_FM71'
session = '4'

# on recharge base commune avec les Hilbert : 
session_common_dir = f"/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP_spikes/{patient}_stim{session}"
bundle = load_common_session_bundle(
    session_common_dir,
    load_hilbert=True,
    hilbert_bands=("theta", "alpha", "beta", "low_gamma", "high_gamma"),
)

# on prépare les spikes depuis le NWB :
spikes = get_nwb(patient, session, root_micro)
units = make_units_dict_from_spikes_object(spikes)

cfg_corr = FRPowerCorrelationConfig(
    bin_width_ms_options=(100.0, 500.0),
    windows_to_correlate=("post",),

    fr_norm_method="logratio",
    epsilon_fr_hz=0.1,

    lfp_value_col="lfp_power_bin_mean",
    bands_to_test=("theta", "alpha", "beta", "low_gamma", "high_gamma"),

    localities_to_include=("local", "distant"),

    correlation_methods=("spearman", "pearson"),
    min_trials_for_corr=6,
    min_finite_pairs_for_corr=6,

    correlation_groupings=(
        "all",
        "group_label",
        "locality",
        "group_label_x_locality",
        "cog_subcategory",
        "cog_subcategory_x_locality",
    ),

    save_trial_bin_tables=True,
    save_fr_table=True,
    save_lfp_table=False,

    save_significant_correlations=True,
    significance_p_col="p_value",
    significance_alpha=0.05,

    make_significant_histograms=True,
    histogram_method="spearman",
    histogram_significance_col="p_value",
    histogram_alpha=0.05,
    histogram_bins=30,

    compression=None,
    verbose=True,
)
out = run_session_fr_power_correlations(
    bundle=bundle,
    units=units,
    out_dir="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP_spikes/results_fr_power_corr",
    cfg=cfg_corr,
)

/media/aube/Aube/Spike-sorting/Data_folders/P119_FM71/P119_FM71_stim4/ .nwb existe deja ? True
[INFO] P119_FM71_stim4: calcul FR biné pour 5 unités
[INFO] P119_FM71_stim4: binning LFP band=theta
[INFO] P119_FM71_stim4: fusion FR×LFP band=theta
[INFO] P119_FM71_stim4: corrélations band=theta


In [ ]:
# pour toutes les sessions dispo : 

from utils_spike_lfp.spike_lfp_power_correlations import (
    FRPowerCorrelationConfig,
    PooledFRPowerCorrelationConfig,
    run_all_available_sessions_fr_power_correlations,
)
root_micro = "/media/aube/Aube/"  # racine compatible avec get_nwb
common_root = "/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP_spikes"
output_root = "/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP_spikes/results_fr_power_corr_all"

cfg_corr = FRPowerCorrelationConfig(
    bin_width_ms_options=(100.0, 500.0),
    windows_to_correlate=("post",),

    fr_norm_method="logratio",
    epsilon_fr_hz=0.1,

    lfp_value_col="lfp_power_bin_mean",
    bands_to_test=("theta", "alpha", "beta", "low_gamma", "high_gamma"),

    localities_to_include=("local", "distant"),
    correlation_methods=("spearman", "pearson"),

    min_trials_for_corr=6,
    min_finite_pairs_for_corr=6,

    correlation_groupings=(
        "all",
        "group_label",
        "locality",
        "group_label_x_locality",
        "cog_subcategory",
        "cog_subcategory_x_locality",
    ),

    save_trial_bin_tables=False,   # conseillé en multi-session pour limiter la taille
    save_fr_table=True,
    save_lfp_table=False,

    save_significant_correlations=True,
    significance_p_col="p_value",
    significance_alpha=0.05,

    make_significant_histograms=True,
    histogram_method="spearman",
    histogram_significance_col="p_value",
    histogram_alpha=0.05,

    compression=None,
    verbose=True,
)

pool_cfg = PooledFRPowerCorrelationConfig(
    common_root=common_root,
    root_micro=root_micro,
    output_root=output_root,
    hilbert_bands=("theta", "alpha", "beta", "low_gamma", "high_gamma"),
    session_cfg=cfg_corr,

    require_existing_nwb=True,
    skip_existing_session_outputs=False,
    recompute_pooled_fdr=True,
    make_pooled_histograms=True,
    verbose=True,
)

out_all = run_all_available_sessions_fr_power_correlations(
    pool_cfg=pool_cfg,
    get_nwb_func=get_nwb,  # important si get_nwb est déjà importée dans ton notebook
)